In [ ]:
from pathlib import Path
import gc
import importlib.util
import os
import subprocess
import sys
import time

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive as colab_drive
    colab_drive.mount("/content/drive", force_remount=False)
    WORKSPACE = Path("/content/drive/MyDrive/Zhong et al. 2025 - Neuromatch Team Workspace")
    CODE = WORKSPACE / "code"
    CACHE = Path("/content/zhong-cache")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy>=1.11,<2"], check=True)
    DATABASE = WORKSPACE / "zhong.duckdb"
else:
    WORKSPACE = next(
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "code").is_dir() and (path / "data" / "cache" / "zhong.duckdb").is_file()
    )
    CODE = WORKSPACE / "code"
    CACHE = WORKSPACE / "data" / "cache"
    DATABASE = CACHE / "zhong.duckdb"
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE / ".matplotlib"))
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import drive
from joiner import Joiner

db = drive.setup(cache=str(CACHE), database=str(DATABASE), mount=False)
assert db.database_path.is_file()
db


In [ ]:
from behavior_neural import (
    frame_trial_metrics,
    position_discriminability,
    behavior_neural_coupling_session,
    trial_behavior,
)

POSITION_EDGES = np.linspace(0.0, 4.0, 21)
OUT = drive.results("tomoya")
OUT.mkdir(parents=True, exist_ok=True)
{"position_bins": len(POSITION_EDGES) - 1, "texture_m": (0.0, 4.0), "target_area": "mHV"}


In [ ]:
manifest = db.query("""
    SELECT b.behavior_session_id, b.behavior_key, b.recording_id,
           b.experiment, b.mouse, b.cohort, e.stage, e.moment, b.trial_count
    FROM behavior_sessions AS b
    JOIN recordings AS r USING (recording_id)
    JOIN experiments AS e USING (experiment)
    WHERE r.has_behavior AND r.has_reduced_neural AND r.has_retinotopy
      AND e.stage = 'train1' AND e.moment IN ('before', 'after')
      AND b.cohort IN ('supervised', 'unsupervised')
    ORDER BY b.cohort, b.mouse, e.moment, b.recording_id
""")
manifest


In [ ]:
assert len(manifest) == 26
db.register("session_manifest", manifest)
db.query("""
    SELECT cohort, moment, COUNT(DISTINCT mouse) AS mice,
           COUNT(DISTINCT behavior_session_id) AS sessions
    FROM session_manifest
    GROUP BY ALL
    ORDER BY cohort, moment
""")


In [ ]:
probe = next(manifest.itertuples(index=False))
db.query("""
    SELECT trial_id, wall_name, stimulus_role, is_rewarded,
           reward_position, sound_position, sound_time, sound_delay_time,
           start_time, end_time
    FROM behavior_trials
    WHERE behavior_session_id = ?
    ORDER BY trial_id
    LIMIT 10
""", [probe.behavior_session_id])


In [ ]:
probe_behavior = trial_behavior(db, probe.behavior_session_id)
probe_behavior[
    [
        "trial_id", "wall_name", "is_rewarded", "reward_position_m",
        "lick_count", "anticipatory_licks_2s", "first_lick_latency_s",
        "previous_reward_position_m",
    ]
].head(12)


In [ ]:
probe_joiner = Joiner(
    db,
    probe.recording_id,
    experiment=probe.experiment,
    behavior_key=probe.behavior_key,
)
probe_joiner.query("""
    SELECT area_group, COUNT(*) AS neurons
    FROM neurons
    GROUP BY area_group
    ORDER BY area_group
""")


In [ ]:
probe_trials = frame_trial_metrics(probe_joiner, POSITION_EDGES)
probe_trials.head(12)


In [ ]:
probe_position = position_discriminability(probe_joiner, POSITION_EDGES)
probe_position


In [ ]:
probe_trials.merge(probe_behavior, on="trial_id")[
    [
        "trial_id", "mean_run_speed", "mean_mhv_z", "mhv_peak_position_m",
        "lick_rate_hz", "previous_reward_position_m",
    ]
].head(12)


In [ ]:
del probe_joiner
gc.collect()

trial_scan = []
position_scan = []
session_scan = []
for index, session in enumerate(manifest.itertuples(index=False), start=1):
    started = time.perf_counter()
    trials, position, summary = behavior_neural_coupling_session(db, session, POSITION_EDGES)
    trial_scan.append(trials)
    position_scan.append(position)
    session_scan.append(summary)
    print(f"{index:02d}/26 {session.behavior_session_id} {len(trials)} trials {time.perf_counter() - started:.1f}s")

trials = pd.concat(trial_scan, ignore_index=True)
positions = pd.concat(position_scan, ignore_index=True)
sessions = pd.DataFrame(session_scan)
trials.shape, positions.shape, sessions.shape


In [ ]:
trials.to_csv(OUT / "trial_behavior_neural.csv", index=False)
positions.to_csv(OUT / "mhv_position_dprime.csv", index=False)
sessions.to_csv(OUT / "session_associations.csv", index=False)
db.register("trial_metrics", trials)
db.register("position_dprime", positions)
db.register("session_metrics", sessions)
db.query("""
    SELECT cohort, moment, COUNT(DISTINCT mouse) AS mice,
           SUM(trials) AS trials,
           AVG(anticipatory_trial_fraction) AS anticipatory_fraction,
           AVG(rho_run_mhv) AS mean_run_mhv_rho,
           AVG(rho_previous_reward_peak) AS mean_previous_reward_peak_rho
    FROM session_metrics
    GROUP BY ALL
    ORDER BY cohort, moment
""")


In [ ]:
db.query("""
    SELECT trial_id, wall_name, reward_position_m, previous_reward_position_m,
           lick_rate_hz, mean_run_speed, mean_mhv_z, mhv_peak_position_m
    FROM trial_metrics
    WHERE behavior_session_id = ?
    ORDER BY trial_id
    LIMIT 15
""", [probe.behavior_session_id])


In [ ]:
mouse_position = db.query("""
    WITH mouse_bins AS (
        SELECT mouse, cohort, moment, position_m,
               AVG(frac_selective) AS frac_selective
        FROM position_dprime
        GROUP BY ALL
    )
    SELECT cohort, moment, position_m, COUNT(*) AS mice,
           AVG(frac_selective) AS frac_selective,
           STDDEV_SAMP(frac_selective) / SQRT(COUNT(*)) AS sem
    FROM mouse_bins
    GROUP BY ALL
    ORDER BY cohort, moment, position_m
""")
mouse_position.head(12)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for (cohort, moment), group in mouse_position.groupby(["cohort", "moment"]):
    axes[0].errorbar(
        group["position_m"], group["frac_selective"],
        yerr=group["sem"].fillna(0), marker="o", label=f"{cohort} {moment}"
    )
anticipatory_pairs = db.query("""
    SELECT cohort, mouse,
           MAX(anticipatory_trial_fraction) FILTER (WHERE moment = 'before') AS before,
           MAX(anticipatory_trial_fraction) FILTER (WHERE moment = 'after') AS after
    FROM session_metrics
    GROUP BY cohort, mouse
    ORDER BY cohort, mouse
""")
for cohort, group in anticipatory_pairs.groupby("cohort"):
    axes[1].scatter(group["before"], group["after"], label=cohort)
axes[0].set(xlabel="position (m)", ylabel="mHV fraction |d′| ≥ 0.3")
axes[0].legend(frameon=False, fontsize=8)
axes[1].plot([0, 1], [0, 1], color="0.7", linestyle="--")
axes[1].set(xlabel="before anticipatory fraction", ylabel="after anticipatory fraction")
axes[1].legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "behavior_neural_coupling.png", dpi=180)
plt.show()


In [ ]:
db.query("""
    SELECT mouse, cohort, moment, rho_run_mhv, rho_lick_mhv,
           rho_previous_reward_peak, anticipatory_trial_fraction,
           first_sustained_selective_position_m, peak_fraction_selective
    FROM session_metrics
    ORDER BY cohort, mouse, moment
""")
